In [1]:
import os, json, subprocess

creds = json.loads(subprocess.check_output(
    ["aws", "configure", "export-credentials", "--profile", "default", "--format", "process"]
))
os.environ["AWS_ACCESS_KEY_ID"] = creds["AccessKeyId"]
os.environ["AWS_SECRET_ACCESS_KEY"] = creds["SecretAccessKey"]
os.environ["AWS_SESSION_TOKEN"] = creds.get("SessionToken", "")

In [2]:
!uv run pyspark --version
!ls $SPARK_HOME/jars | grep -i hadoop-aws

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.1.1
      /_/
                        
Using Scala version 2.13.17, OpenJDK 64-Bit Server VM, 21.0.12
Branch HEAD
Compiled by user runner on 2026-01-10T20:56:35Z
Revision c0690c763bafabd08e7079d1137fa0a769a05bae
Url https://github.com/apache/spark.git
Type --help for more information.
hadoop-aws-3.4.2.jar


In [3]:
import os
from pyspark.sql import SparkSession, functions as F

BUCKET = os.environ["KUBERNETES_NAMESPACE"]
ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0"

spark = (
    SparkSession.builder
    .appName("lab-iceberg-pyspark")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{os.environ['AWS_S3_ENDPOINT']}")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.session.token", os.environ.get("AWS_SESSION_TOKEN", ""))
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.jars.packages", ICEBERG_PACKAGE)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", f"s3a://{BUCKET}/warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.sql("SHOW NAMESPACES IN lab").show()

:: loading settings :: url = jar:file:/usr/local/lib/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/onyxia/.ivy2.5.2/cache
The jars for the packages stored in: /home/onyxia/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3cca7484-ffbf-4e23-8d79-1d30140e58ce;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
:: resolution report :: resolve 198ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   | 

+---------+
|namespace|
+---------+
|   silver|
+---------+



In [4]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lab.silver")

spark.read.parquet(f"s3a://{BUCKET}/silver/spark/orders.parquet") \
    .writeTo("lab.silver.orders").createOrReplace()
spark.read.parquet(f"s3a://{BUCKET}/silver/spark/users.parquet") \
    .writeTo("lab.silver.users").createOrReplace()

spark.sql("SELECT count(*) AS orders, sum(quantity) AS quantity FROM lab.silver.orders").show()

+------+--------+
|orders|quantity|
+------+--------+
|  2829|    8615|
+------+--------+



In [5]:
!aws s3 --profile default ls "s3://{BUCKET}/warehouse/silver/orders/" --recursive

2026-11-06 00:25:03      84055 warehouse/silver/orders/data/00000-2-626035f5-80f3-4369-9585-de24c5a81e73-0-00001.parquet
2026-11-06 00:11:21      84055 warehouse/silver/orders/data/00000-2-f2a86e30-f49a-4e0b-affa-7db607f5b821-0-00001.parquet
2026-11-06 00:11:21       7344 warehouse/silver/orders/metadata/2cae6ce4-1a4b-439e-8733-e1b37e99dc6c-m0.avro
2026-11-06 00:25:03       7348 warehouse/silver/orders/metadata/4990b556-5011-4b71-ac21-cc5a5cb6e625-m0.avro
2026-11-06 00:25:04       4458 warehouse/silver/orders/metadata/snap-1082701254134585756-1-4990b556-5011-4b71-ac21-cc5a5cb6e625.avro
2026-11-06 00:11:21       4456 warehouse/silver/orders/metadata/snap-2883126036010309816-1-2cae6ce4-1a4b-439e-8733-e1b37e99dc6c.avro
2026-11-06 00:11:22       1930 warehouse/silver/orders/metadata/v1.metadata.json
2026-11-06 00:25:04       2961 warehouse/silver/orders/metadata/v2.metadata.json
2026-11-06 00:25:04          1 warehouse/silver/orders/metadata/version-hint.text


In [6]:
spark.sql("SELECT * FROM lab.silver.orders.snapshots").show(truncate=False)
spark.sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM lab.silver.orders.files
""").show(truncate=False)

+-----------------------+-------------------+---------+---------+---------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                              |summary                                            

In [7]:
stg_orders = spark.table("lab.silver.orders")
stg_orders.show(5)

new_orders = stg_orders.limit(3).withColumn("order_id", F.expr("uuid()"))
new_orders.writeTo("lab.silver.orders").append()

spark.sql("""
    SELECT snapshot_id, operation, committed_at
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

spark.sql("SELECT count(*) FROM lab.silver.orders").show()

+--------------------+--------------------+--------------------+--------+---------+
|            order_id|             user_id|          ordered_at|quantity|  product|
+--------------------+--------------------+--------------------+--------+---------+
|00060d5a-02b3-494...|f53d4330-cdda-44b...|2020-01-28 12:31:...|       2|   cookie|
|0011be30-e268-492...|5f733a3e-5d28-4aa...|2020-04-18 22:51:...|       5|    bread|
|002394e2-2ad9-468...|90164161-cfa7-41c...|2020-02-24 01:03:...|       4|   cookie|
|0041f5d4-9549-463...|aab612c9-415d-474...|2020-03-17 18:29:...|       2|croissant|
|00684d9c-d2bf-4c3...|97fa7f04-8363-400...|2020-03-30 13:36:...|       4|    bread|
+--------------------+--------------------+--------------------+--------+---------+
only showing top 5 rows
+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|2883126036010309816|overwrite|2026-11-06 00:11:21.8

In [8]:
spark.sql("ALTER TABLE lab.silver.orders ADD COLUMN source STRING")
spark.sql("DESCRIBE lab.silver.orders").show()

spark.sql("""
    INSERT INTO lab.silver.orders
    SELECT order_id, user_id, ordered_at, quantity, product, 'spark'
    FROM lab.silver.orders
    LIMIT 1
""")

!aws s3 --profile default ls "s3://{BUCKET}/warehouse/silver/orders/metadata/"

+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|  order_id|   string|   NULL|
|   user_id|   string|   NULL|
|ordered_at|timestamp|   NULL|
|  quantity|      int|   NULL|
|   product|   string|   NULL|
|    source|   string|   NULL|
+----------+---------+-------+

2026-11-06 00:25:39       7334 136d6678-d8f1-44b5-9f9f-110a84f52a9f-m0.avro
2026-11-06 00:11:21       7344 2cae6ce4-1a4b-439e-8733-e1b37e99dc6c-m0.avro
2026-11-06 00:25:03       7348 4990b556-5011-4b71-ac21-cc5a5cb6e625-m0.avro
2026-11-06 00:25:51       7367 622f587a-7994-4ebc-a140-0dfe197cb539-m0.avro
2026-11-06 00:25:04       4458 snap-1082701254134585756-1-4990b556-5011-4b71-ac21-cc5a5cb6e625.avro
2026-11-06 00:11:21       4456 snap-2883126036010309816-1-2cae6ce4-1a4b-439e-8733-e1b37e99dc6c.avro
2026-11-06 00:25:51       4584 snap-7579727681780096917-1-622f587a-7994-4ebc-a140-0dfe197cb539.avro
2026-11-06 00:25:39       4533 snap-8751582675676698646-1-136d6678-d8f1-44b5-9f9f-110

In [9]:
!aws s3 --profile default ls "s3://{BUCKET}/silver/spark/orders.parquet/" --recursive
!aws s3 --profile default ls "s3://{BUCKET}/warehouse/silver/orders/" --recursive

2026-10-26 20:41:11      11810 silver/spark/orders.parquet/_SUCCESS
2026-10-26 20:41:11     139556 silver/spark/orders.parquet/part-00000-5e88aeff-5842-48ba-9919-e38efa1e7b9f-c000.snappy.parquet
2026-11-06 00:25:39       1954 warehouse/silver/orders/data/00000-12-a593e0a9-facb-4f79-a26b-1597d027acfc-0-00001.parquet
2026-11-06 00:25:51       2201 warehouse/silver/orders/data/00000-17-83fb079a-c06f-4681-9b04-a9f0eedc71e4-0-00001.parquet
2026-11-06 00:25:03      84055 warehouse/silver/orders/data/00000-2-626035f5-80f3-4369-9585-de24c5a81e73-0-00001.parquet
2026-11-06 00:11:21      84055 warehouse/silver/orders/data/00000-2-f2a86e30-f49a-4e0b-affa-7db607f5b821-0-00001.parquet
2026-11-06 00:25:39       7334 warehouse/silver/orders/metadata/136d6678-d8f1-44b5-9f9f-110a84f52a9f-m0.avro
2026-11-06 00:11:21       7344 warehouse/silver/orders/metadata/2cae6ce4-1a4b-439e-8733-e1b37e99dc6c-m0.avro
2026-11-06 00:25:03       7348 warehouse/silver/orders/metadata/4990b556-5011-4b71-ac21-cc5a5cb6e625-

In [10]:
users = spark.table("lab.silver.users")
users.printSchema()

users.limit(3).withColumn("user_id", F.expr("uuid()")).writeTo("lab.silver.users").append()
spark.sql("SELECT snapshot_id, operation FROM lab.silver.users.snapshots ORDER BY committed_at").show(truncate=False)

root
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- email: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)

+-------------------+---------+
|snapshot_id        |operation|
+-------------------+---------+
|8481679296752731327|overwrite|
|2682976800235037293|overwrite|
|3404946714129719595|append   |
+-------------------+---------+



In [11]:
snaps = spark.sql("SELECT snapshot_id FROM lab.silver.users.snapshots ORDER BY committed_at").collect()
spark.sql(f"SELECT count(*) FROM lab.silver.users VERSION AS OF {snaps[0]['snapshot_id']}").show()
spark.sql("SELECT count(*) FROM lab.silver.users").show()

+--------+
|count(1)|
+--------+
|      50|
+--------+

+--------+
|count(1)|
+--------+
|      53|
+--------+



In [12]:
spark.sql("ALTER TABLE lab.silver.users ADD COLUMN country STRING")

users = spark.table("lab.silver.users")
users.limit(1).withColumn("user_id", F.expr("uuid()")) \
    .withColumn("country", F.lit("FR")) \
    .writeTo("lab.silver.users").append()

spark.sql("SELECT country, count(*) FROM lab.silver.users GROUP BY country").show()

+-------+--------+
|country|count(1)|
+-------+--------+
|   NULL|      53|
|     FR|       1|
+-------+--------+



In [13]:
!aws s3 --profile default ls "s3://{BUCKET}/warehouse/silver/users/" --recursive

2026-11-06 00:26:11       3309 warehouse/silver/users/data/00000-19-4fcb13b5-d6f1-42a0-b408-49e5114c4f1b-0-00001.parquet
2026-11-06 00:26:29       3544 warehouse/silver/users/data/00000-29-94e7a10c-573c-4e70-9b39-21ff1467fd87-0-00001.parquet
2026-11-06 00:11:24       7435 warehouse/silver/users/data/00000-5-324c0daf-c816-44ba-883e-9bc16af6462e-0-00001.parquet
2026-11-06 00:25:06       7435 warehouse/silver/users/data/00000-5-a319e79e-7df8-43c5-a120-68a765a99cf5-0-00001.parquet
2026-11-06 00:26:29       7726 warehouse/silver/users/metadata/06617369-0a95-48d7-9a61-aa9afb07a8e9-m0.avro
2026-11-06 00:26:11       7739 warehouse/silver/users/metadata/2c1d6ac6-29ca-436e-b13a-67724b16f06d-m0.avro
2026-11-06 00:11:24       7738 warehouse/silver/users/metadata/2cb97835-790c-4d49-bc01-65322ce5dc38-m0.avro
2026-11-06 00:25:06       7740 warehouse/silver/users/metadata/72f0c560-70c8-4f5e-ba44-0ed978e143d3-m0.avro
2026-11-06 00:25:06       4453 warehouse/silver/users/metadata/snap-268297680023503729